# Stage 6 - RAG (retrieval augmented generation)

Now we put the comments behind a local LLM so we can ask questions and get answers that are
*grounded* in what people actually said (not the model making stuff up).

Flow: embed every comment -> store in a vector db (Chroma) -> for a question, retrieve the most
relevant comments -> stuff them into the prompt -> let Ollama answer.

The heavy lifting lives in `../src/rag_pipeline.py` so the dashboard can reuse it. Here we just
drive it and show outputs.

Needs: `ollama serve` running + the model pulled (`ollama pull llama3.2:3b`).

In [1]:
import sys
sys.path.append("../src")
import rag_pipeline as rag

## Build the vector index

Embeds all ~20k comments once and saves to `../chroma_db`. Re-running is cheap (it reuses it).

In [2]:
col = rag.build_index()
print("comments in index:", col.count())

c:\Users\Saeed\Documents\370\notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4681.70it/s]


  indexed 512/18771
  indexed 1024/18771
  indexed 1536/18771
  indexed 2048/18771
  indexed 2560/18771
  indexed 3072/18771
  indexed 3584/18771
  indexed 4096/18771
  indexed 4608/18771
  indexed 5120/18771
  indexed 5632/18771
  indexed 6144/18771
  indexed 6656/18771
  indexed 7168/18771
  indexed 7680/18771
  indexed 8192/18771
  indexed 8704/18771
  indexed 9216/18771
  indexed 9728/18771
  indexed 10240/18771
  indexed 10752/18771
  indexed 11264/18771
  indexed 11776/18771
  indexed 12288/18771
  indexed 12800/18771
  indexed 13312/18771
  indexed 13824/18771
  indexed 14336/18771
  indexed 14848/18771
  indexed 15360/18771
  indexed 15872/18771
  indexed 16384/18771
  indexed 16896/18771
  indexed 17408/18771
  indexed 17920/18771
  indexed 18432/18771
  indexed 18771/18771
done - 18771 comments in chroma
comments in index: 18771


## Try retrieval on its own

Before adding the LLM, check that semantic search returns sensible comments.

In [3]:
hits = rag.retrieve("is the battery life good?", k=5)
for doc, meta in hits:
    print(f"[{meta['sentiment']}] {doc[:120]}")

[positive] is the battery life good
[neutral] how is the battery life ?
[positive] is battery any good?
[positive] really battery life is best
[positive] is the battery any better


In [4]:
# hybrid retrieval - same query but only negative comments
hits = rag.retrieve("battery", k=5, sentiment="negative")
for doc, meta in hits:
    print("-", doc[:120])

- battery sucks
- the battery sucks
- the alone for the designer and the battery
- you forget the bigger battery
- the battery is so so bad


## Question answering

Retrieve + answer. The prompt tells the model to only use the comments.

## Retrieval modes

We support four modes in `rag_pipeline.py`:
- **semantic** - embedding search in chroma
- **lexical** - tf-idf cosine similarity
- **metadata** - semantic + sentiment/topic filter
- **hybrid** - weighted mix (default for QA)


In [ ]:
q = "battery life"
for mode in ["semantic", "lexical", "hybrid"]:
    hits = rag.retrieve(q, k=3, mode=mode)
    print(f"\n=== {mode} ===")
    for doc, meta in hits:
        print(f"[{meta['sentiment']}] {doc[:100]}")

print("\n=== metadata (negative only) ===")
for doc, meta in rag.retrieve(q, k=3, mode="metadata", sentiment="negative"):
    print(f"[{meta['sentiment']}] {doc[:100]}")


In [5]:
answer, sources = rag.answer_question("what do people think about the battery life?")
print(answer)
print("\n--- grounded on ---")
for doc, meta in sources[:3]:
    print("*", doc[:100])

Based on the comments provided, here's a summary of what people think about the battery life:

* Some people are curious or concerned about the battery life (e.g., "how is the battery life ?", "is the battery life good", "battery health ?").
* A few people seem to be making guesses or expressing optimism about the battery life (e.g., "really battery life is best", "let me guess, better battery life").

There doesn't appear to be any direct statement about the actual battery life or its quality.

--- grounded on ---
* how is the battery life ?
* is the battery life good
* really battery life is best


In [6]:
answer, _ = rag.answer_question("did anyone complain about the price?")
print(answer)

Yes, some people complained about the price, and one person even said "yeah, it is terrible... buy 1 for me". However, others didn't seem to mind or thought it wasn't that bad.


## Summarization

Same retrieval, different prompt - pulls more comments and summarises them.

In [7]:
summary, _ = rag.summarize("camera quality")
print(summary)

Here is a 4-5 sentence summary covering the main points, overall mood, and anything that keeps coming up:

The comments revolve around testing the camera quality of a phone model. Many users are inquiring about whether the camera has improved or if any other applications have been updated to showcase its capabilities. The topic of camera quality is consistently brought up, with some users asking for specific comparisons (e.g., 4:36 per camera quality) and others expressing their opinions on the camera's performance. A sense of curiosity and interest in the phone's camera features is evident throughout the comments. Overall, the mood is inquisitive and enthusiastic, with users eager to learn more about the phone's camera capabilities.
